### Uploading local files to Colab

To upload files from your local computer to the Colab environment, you can use the `files.upload()` utility from `google.colab`. When you run the code, it will present an upload button allowing you to browse and select files from your local file system.

In [1]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Saving LLM_Interview_Dataset-v2.csv to LLM_Interview_Dataset-v2 (2).csv
User uploaded file "LLM_Interview_Dataset-v2 (2).csv" with length 395812 bytes


Once uploaded, the files will be available in the current Colab runtime session. Keep in mind that these files are temporary and will be deleted when the runtime is reset.

In [2]:
!pip install transformers peft trl torch pandas datasets accelerate

In [3]:
!pip install --upgrade torchao peft trl transformers accelerate

In [7]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# 1. Load and format dataset
df = pd.read_csv("LLM_Interview_Dataset-v2.csv")
df['text'] = df.apply(
    lambda row: f"Domain: {row['Domain']}\nDifficulty: {row['Difficulty_Level']}\nQuestion: {row['Question']}\nIdeal Answer: {row['Reference_Answer']}",
    axis=1
)
dataset = Dataset.from_pandas(df[['text']])

# 2. Load Base Model
model_id = "Qwen/Qwen1.5-1.8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype=torch.float16)

# 3. Apply LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# 4. Set Arguments via SFTConfig (Using 'max_length' instead of 'max_seq_length')
sft_config = SFTConfig(
    output_dir="./results",
    dataset_text_field="text",
    max_length=256,                 # <-- This is the exact fix
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=10,
)

# 5. Train
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
)

trainer.train()

# 6. Save fine-tuned weights
trainer.model.save_pretrained("./fine_tuned_interviewer")
tokenizer.save_pretrained("./fine_tuned_interviewer")
print("Training complete! Download the './fine_tuned_interviewer' folder.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.746224
20,2.571541
30,2.537285
40,2.357299
50,2.276603
60,2.004947
70,1.961206
80,1.850322
90,1.660386
100,1.574475


Training complete! Download the './fine_tuned_interviewer' folder.


In [8]:
!zip -r fine_tuned_interviewer.zip ./fine_tuned_interviewer

  adding: fine_tuned_interviewer/ (stored 0%)
  adding: fine_tuned_interviewer/chat_template.jinja (deflated 46%)
  adding: fine_tuned_interviewer/adapter_model.safetensors (deflated 8%)
  adding: fine_tuned_interviewer/adapter_config.json (deflated 59%)
  adding: fine_tuned_interviewer/tokenizer_config.json (deflated 50%)
  adding: fine_tuned_interviewer/README.md (deflated 65%)
  adding: fine_tuned_interviewer/tokenizer.json (deflated 81%)
